# Manual Testing: Complete 4-Agent Pipeline with SpecExtractorAgent

This notebook provides hands-on testing of the complete pipeline using real API keys:

## Pipeline Flow
1. **QueryOrchestrator** - Parses user query → structured search query + intent
2. **TavilyRetriever** - Executes Tavily two-step process (search → extract)
3. **CredibilityFilter** - Scores and filters results by source credibility
4. **SpecExtractor** - Extracts structured product specifications (universal)
5. **Analysis Tools** - Interactive analysis of extracted specifications

## SpecExtractor Features
- **Universal Category Detection**: Works with ANY product type (electronics, kitchen, fashion, toys, automotive, books, etc.)
- **Dynamic Spec Extraction**: Pattern recognition + LLM enhancement
- **Unit Normalization**: Standardizes weights, dimensions, memory, etc.
- **Coverage Calculation**: Measures extraction quality (target: 60%+)

## Requirements
- Real OPENAI_API_KEY and TAVILY_API_KEY in .env file
- Backend dependencies installed

## Setup and Imports

In [ ]:
import os
import sys
import asyncio
import json
from datetime import datetime, timezone, timedelta
from pprint import pprint
import pandas as pd
from urllib.parse import urlparse

# Add backend to path
sys.path.append('..')

# Load environment variables
from dotenv import load_dotenv
load_dotenv('../.env')

print("✅ Environment loaded")
print(f"OPENAI_API_KEY: {'✅ Set' if os.getenv('OPENAI_API_KEY') else '❌ Missing'}")
print(f"TAVILY_API_KEY: {'✅ Set' if os.getenv('TAVILY_API_KEY') else '❌ Missing'}")

In [ ]:
# Import our agents and state management
from app.agents.query_orchestrator_agent import QueryOrchestratorAgent
from app.agents.tavily_retriever_agent import TavilyRetrieverAgent
from app.agents.credibility_filter_agent import CredibilityFilterAgent
from app.agents.spec_extractor_agent import SpecExtractorAgent
from app.agents.state import create_initial_state, get_state_summary
from app.config import settings

print("✅ Agents imported successfully")
print(f"Environment: {settings.ENVIRONMENT}")
print(f"OpenAI Model: {settings.OPENAI_MODEL}")

## Initialize Agents

In [ ]:
# Create agent instances
query_agent = QueryOrchestratorAgent()
tavily_agent = TavilyRetrieverAgent()
credibility_agent = CredibilityFilterAgent()
spec_agent = SpecExtractorAgent()

print("✅ Agents initialized:")
print(f"- {query_agent.name} (OpenAI: {query_agent.llm.model_name})")
print(f"- {tavily_agent.name} (Client: {type(tavily_agent.tavily_client).__name__})")
print(f"- {credibility_agent.name} (Threshold: {credibility_agent.credibility_threshold})")
print(f"- {spec_agent.name} (LLM: {spec_agent.llm.model_name})")

## Test Configuration

In [ ]:
# Test Tavily configuration
config_test = await tavily_agent.test_configuration()
print("🔧 Tavily Configuration:")
pprint(config_test)

# Show SpecExtractor configuration
print("\n🎯 SpecExtractor Configuration:")
print(f"  Coverage threshold: {spec_agent.min_coverage_threshold}")
print(f"  LLM enhancement threshold: {spec_agent.llm_enhancement_threshold}")
print(f"  Category detection: {len(spec_agent.category_detector.CATEGORY_KEYWORDS)} categories")
print(f"  Pattern types: {len(spec_agent.spec_extractor.UNIVERSAL_SPEC_PATTERNS)} universal patterns")

## Complete Pipeline Test Functions

In [ ]:
async def test_complete_pipeline(raw_query: str, run_id: str = None):
    """
    Complete 4-agent pipeline test: QueryOrchestrator → TavilyRetriever → CredibilityFilter → SpecExtractor
    """
    if not run_id:
        run_id = f"manual_test_{datetime.now().strftime('%H%M%S')}"
    
    print(f"\n🚀 Testing Complete 4-Agent Pipeline: '{raw_query}'")
    print("=" * 80)
    
    # Create initial state
    state = create_initial_state(raw_query=raw_query, run_id=run_id)
    print(f"📋 Initial state created (run_id: {run_id})")
    
    try:
        # Step 1: QueryOrchestrator
        print("\n1️⃣ QueryOrchestrator Processing...")
        start_time = datetime.now()
        state = await query_agent.process(state)
        query_time = (datetime.now() - start_time).total_seconds()
        
        search_query = state.get("search_query")
        if search_query:
            print(f"✅ Query parsed successfully ({query_time:.2f}s)")
            print(f"   Intent: {search_query.intent}")
            print(f"   Category: {search_query.category}")
            print(f"   Normalized: '{search_query.normalized_query}'")
            if search_query.budget_max:
                print(f"   Budget: ${search_query.budget_max}")
            if search_query.constraints:
                print(f"   Constraints: {search_query.constraints}")
        else:
            print("❌ Query parsing failed")
            return state
        
        # Step 2: TavilyRetriever
        print("\n2️⃣ TavilyRetriever Processing...")
        start_time = datetime.now()
        state = await tavily_agent.process(state)
        tavily_time = (datetime.now() - start_time).total_seconds()
        
        search_results = state.get("raw_search_results", [])
        extracted_content = state.get("extracted_content", [])
        coverage_score = state.get("coverage_score", 0.0)
        
        print(f"✅ Tavily processing completed ({tavily_time:.2f}s)")
        print(f"   Search results: {len(search_results)}")
        print(f"   Extracted content: {len(extracted_content)}")
        print(f"   Coverage score: {coverage_score:.2f}")
        
        # Step 3: CredibilityFilter
        print("\n3️⃣ CredibilityFilter Processing...")
        start_time = datetime.now()
        state = await credibility_agent.process(state)
        credibility_time = (datetime.now() - start_time).total_seconds()
        
        filtered_results = state.get("credibility_filtered_results", [])
        
        print(f"✅ Credibility filtering completed ({credibility_time:.2f}s)")
        print(f"   Filtered results: {len(filtered_results)} (from {len(search_results)})")
        
        if filtered_results:
            scores = [r.get("credibility_score", 0) for r in filtered_results]
            avg_score = sum(scores) / len(scores)
            print(f"   Average credibility: {avg_score:.3f}")
            print(f"   Score range: {min(scores):.3f} - {max(scores):.3f}")
        
        # Step 4: SpecExtractor (NEW!)
        print("\n4️⃣ SpecExtractor Processing...")
        start_time = datetime.now()
        state = await spec_agent.process(state)
        spec_time = (datetime.now() - start_time).total_seconds()
        
        structured_products = state.get("structured_products", [])
        
        print(f"✅ Specification extraction completed ({spec_time:.2f}s)")
        print(f"   Structured products: {len(structured_products)}")
        
        if structured_products:
            coverages = [p.get("extraction_coverage", 0) for p in structured_products]
            avg_coverage = sum(coverages) / len(coverages)
            print(f"   Average coverage: {avg_coverage:.1%}")
            print(f"   Coverage range: {min(coverages):.1%} - {max(coverages):.1%}")
            
            # Show categories detected
            categories = [p.get("category", "unknown") for p in structured_products]
            unique_categories = list(set(categories))
            print(f"   Categories detected: {', '.join(unique_categories)}")
        
        # Show agent execution summary
        summary = get_state_summary(state)
        total_time = query_time + tavily_time + credibility_time + spec_time
        print(f"\n📊 Pipeline Summary:")
        print(f"   Total time: {total_time:.2f}s")
        print(f"   Agents completed: {summary['progress']['agents_completed']}")
        print(f"   Total cost: ${summary['progress']['total_cost_usd']:.4f}")
        
        return state
        
    except Exception as e:
        print(f"❌ Pipeline error: {e}")
        import traceback
        traceback.print_exc()
        return state

def analyze_extracted_specifications(state: dict, detailed: bool = True):
    """
    Analyze extracted product specifications in detail
    """
    structured_products = state.get("structured_products", [])
    
    if not structured_products:
        print("No structured products to analyze")
        return
    
    print(f"\n🔍 Specification Analysis ({len(structured_products)} products):")
    print("=" * 70)
    
    for i, product in enumerate(structured_products, 1):
        title = product.get("title", "No title")
        category = product.get("category", "unknown")
        brand = product.get("brand", "N/A")
        price = product.get("price")
        currency = product.get("currency", "")
        coverage = product.get("extraction_coverage", 0)
        method = product.get("extraction_method", "unknown")
        specs = product.get("specs", {})
        
        print(f"\n{i}. {title[:60]}...")
        print(f"   🏷️  Category: {category}")
        print(f"   🏢 Brand: {brand}")
        print(f"   💰 Price: ${price} {currency}" if price else "   💰 Price: N/A")
        print(f"   📊 Coverage: {coverage:.1%}")
        print(f"   🔧 Method: {method}")
        
        if detailed and specs:
            print(f"   📋 Specifications ({len(specs)} extracted):")
            for key, value in specs.items():
                print(f"      • {key}: {value}")
        elif specs:
            print(f"   📋 Specifications: {len(specs)} extracted (use detailed=True to see all)")

def compare_category_detection(state: dict):
    """
    Compare query category vs detected product categories
    """
    search_query = state.get("search_query")
    structured_products = state.get("structured_products", [])
    
    if not search_query or not structured_products:
        print("Insufficient data for category comparison")
        return
    
    query_category = search_query.category
    product_categories = [p.get("category", "unknown") for p in structured_products]
    
    print(f"\n🎯 Category Detection Comparison:")
    print("=" * 50)
    print(f"Query Category: {query_category}")
    print(f"Product Categories: {list(set(product_categories))}")
    
    # Check consistency
    matches = sum(1 for cat in product_categories if cat == query_category)
    consistency = matches / len(product_categories) * 100 if product_categories else 0
    
    print(f"Category Consistency: {consistency:.1f}% ({matches}/{len(product_categories)})")

def show_extraction_methods(state: dict):
    """
    Show which extraction methods were used
    """
    structured_products = state.get("structured_products", [])
    
    if not structured_products:
        return
    
    methods = {}
    for product in structured_products:
        method = product.get("extraction_method", "unknown")
        methods[method] = methods.get(method, 0) + 1
    
    print(f"\n🔧 Extraction Methods Used:")
    print("=" * 40)
    
    for method, count in methods.items():
        percentage = count / len(structured_products) * 100
        print(f"   {method}: {count} products ({percentage:.1f}%)")

def coverage_distribution_analysis(state: dict):
    """
    Analyze coverage score distribution
    """
    structured_products = state.get("structured_products", [])
    
    if not structured_products:
        return
    
    coverages = [p.get("extraction_coverage", 0) for p in structured_products]
    
    print(f"\n📊 Coverage Distribution Analysis:")
    print("=" * 50)
    
    # Coverage ranges
    ranges = {
        "Excellent (80%+)": [c for c in coverages if c >= 0.8],
        "Good (60-80%)": [c for c in coverages if 0.6 <= c < 0.8],
        "Fair (40-60%)": [c for c in coverages if 0.4 <= c < 0.6],
        "Poor (<40%)": [c for c in coverages if c < 0.4]
    }
    
    for range_name, range_coverages in ranges.items():
        count = len(range_coverages)
        percentage = count / len(coverages) * 100 if coverages else 0
        print(f"   {range_name}: {count} products ({percentage:.1f}%)")
    
    # Statistics
    if coverages:
        avg_coverage = sum(coverages) / len(coverages)
        min_coverage = min(coverages)
        max_coverage = max(coverages)
        
        print(f"\n📈 Statistics:")
        print(f"   Average: {avg_coverage:.1%}")
        print(f"   Range: {min_coverage:.1%} - {max_coverage:.1%}")
        print(f"   Target (60%+): {'✅ Met' if avg_coverage >= 0.6 else '❌ Below target'}")

## Interactive Testing

Now you can test different queries manually and analyze the specification extraction results!

### Test 1: Electronics - Gaming Laptop

In [ ]:
# Test electronics category with complex specifications
electronics_state = await test_complete_pipeline("gaming laptop RTX 4070 32GB RAM under $2000")

In [ ]:
# Analyze the extracted specifications
analyze_extracted_specifications(electronics_state, detailed=True)
compare_category_detection(electronics_state)
show_extraction_methods(electronics_state)
coverage_distribution_analysis(electronics_state)

### Test 2: Kitchen Appliances - Blender

In [ ]:
# Test kitchen category with different specification types
kitchen_state = await test_complete_pipeline("high power blender for smoothies 1000+ watts")

In [ ]:
# Analyze kitchen appliance specifications
analyze_extracted_specifications(kitchen_state, detailed=True)
compare_category_detection(kitchen_state)
coverage_distribution_analysis(kitchen_state)

### Test 3: Fashion - Running Shoes

In [ ]:
# Test fashion category with size/material specifications
fashion_state = await test_complete_pipeline("Nike running shoes men size 10 breathable")

In [ ]:
# Analyze fashion product specifications
analyze_extracted_specifications(fashion_state, detailed=True)
compare_category_detection(fashion_state)

### Test 4: Toys - LEGO Sets

In [ ]:
# Test toys category with age/piece specifications
toys_state = await test_complete_pipeline("LEGO Creator sets for adults 1000+ pieces")

In [ ]:
# Analyze toy specifications
analyze_extracted_specifications(toys_state, detailed=True)
compare_category_detection(toys_state)

### Test 5: Automotive - Car Tires

In [ ]:
# Test automotive category
auto_state = await test_complete_pipeline("all season car tires 235/60R18")

In [ ]:
# Analyze automotive specifications
analyze_extracted_specifications(auto_state, detailed=True)
compare_category_detection(auto_state)

### Custom Query Testing

Use this cell to test your own queries and analyze the specification extraction:

In [ ]:
# Office supplies category test
custom_query = "Office Table with comfortable chair under $1200"
custom_result = await test_complete_pipeline(custom_query)

In [ ]:
# Analyze your custom results
analyze_extracted_specifications(custom_result, detailed=True)
compare_category_detection(custom_result)
show_extraction_methods(custom_result)
coverage_distribution_analysis(custom_result)

## Advanced Analysis: Cross-Category Comparison

In [ ]:
def cross_category_analysis():
    """
    Analyze SpecExtractor performance across different product categories
    """
    print("🎯 Cross-Category SpecExtractor Analysis:")
    print("=" * 60)
    
    # Collect all results from previous tests
    test_results = []
    
    # Add results if they exist
    for var_name, state_var in [
        ("Electronics", globals().get('electronics_state')),
        ("Kitchen", globals().get('kitchen_state')),
        ("Fashion", globals().get('fashion_state')),
        ("Toys", globals().get('toys_state')),
        ("Automotive", globals().get('auto_state')),
        ("Custom", globals().get('custom_result')),
    ]:
        if state_var and state_var.get("structured_products"):
            products = state_var["structured_products"]
            avg_coverage = sum(p.get("extraction_coverage", 0) for p in products) / len(products)
            categories = list(set(p.get("category", "unknown") for p in products))
            
            test_results.append({
                "query_type": var_name,
                "products_count": len(products),
                "avg_coverage": avg_coverage,
                "categories_detected": categories
            })
    
    if not test_results:
        print("No test results available. Run some pipeline tests first.")
        return
    
    # Display results
    for result in test_results:
        print(f"\n📊 {result['query_type']}:")
        print(f"   Products extracted: {result['products_count']}")
        print(f"   Average coverage: {result['avg_coverage']:.1%}")
        print(f"   Categories detected: {', '.join(result['categories_detected'])}")
    
    # Overall statistics
    total_products = sum(r['products_count'] for r in test_results)
    overall_avg = sum(r['avg_coverage'] * r['products_count'] for r in test_results) / total_products
    all_categories = set()
    for r in test_results:
        all_categories.update(r['categories_detected'])
    
    print(f"\n🏆 Overall Performance:")
    print(f"   Total products processed: {total_products}")
    print(f"   Overall average coverage: {overall_avg:.1%}")
    print(f"   Categories successfully detected: {len(all_categories)}")
    print(f"   All categories: {', '.join(sorted(all_categories))}")
    
    if overall_avg >= 0.6:
        print(f"\n✅ SUCCESS: Exceeds 60% coverage target!")
    else:
        print(f"\n⚠️ BELOW TARGET: {overall_avg:.1%} < 60% target")

cross_category_analysis()

## Performance Testing

In [ ]:
# Performance test with multiple diverse queries
performance_queries = [
    "mechanical keyboard RGB gaming",
    "stand mixer for baking bread", 
    "winter jacket waterproof men",
    "board games for family night",
    "programming books Python beginner"
]

performance_results = []

print("⚡ Performance Testing Across Categories:")
print("=" * 50)

for i, query in enumerate(performance_queries, 1):
    print(f"\n{i}. Testing: '{query}'")
    start_time = datetime.now()
    
    result = await test_complete_pipeline(query, f"perf_test_{i}")
    
    total_time = (datetime.now() - start_time).total_seconds()
    summary = get_state_summary(result)
    
    structured_products = result.get("structured_products", [])
    avg_coverage = sum(p.get("extraction_coverage", 0) for p in structured_products) / max(len(structured_products), 1)
    
    categories = list(set(p.get("category", "unknown") for p in structured_products))
    
    performance_results.append({
        "query": query,
        "time_seconds": total_time,
        "cost_usd": summary['progress']['total_cost_usd'],
        "products_extracted": len(structured_products),
        "avg_coverage": avg_coverage,
        "categories": categories
    })

print("\n📊 Performance Summary:")
print("=" * 70)
df = pd.DataFrame(performance_results)
print(df[['query', 'time_seconds', 'products_extracted', 'avg_coverage']].to_string(index=False, float_format=lambda x: f'{x:.2f}'))

print(f"\n🎯 Performance Metrics:")
print(f"   Average time: {df['time_seconds'].mean():.1f}s")
print(f"   Average cost: ${df['cost_usd'].mean():.4f}")
print(f"   Average products per query: {df['products_extracted'].mean():.1f}")
print(f"   Average coverage: {df['avg_coverage'].mean():.1%}")
print(f"   Time per product: {(df['time_seconds'].sum() / df['products_extracted'].sum()):.1f}s")

## LLM Enhancement Analysis

In [ ]:
def analyze_llm_enhancement_usage():
    """
    Analyze when and how LLM enhancement was used
    """
    print("🤖 LLM Enhancement Usage Analysis:")
    print("=" * 50)
    
    all_products = []
    
    # Collect products from all test states
    for state_var in [globals().get(name) for name in [
        'electronics_state', 'kitchen_state', 'fashion_state', 
        'toys_state', 'auto_state', 'custom_result'
    ]]:
        if state_var and state_var.get("structured_products"):
            all_products.extend(state_var["structured_products"])
    
    if not all_products:
        print("No products available for analysis")
        return
    
    # Analyze extraction methods
    methods = {}
    coverage_by_method = {}
    
    for product in all_products:
        method = product.get("extraction_method", "unknown")
        coverage = product.get("extraction_coverage", 0)
        
        methods[method] = methods.get(method, 0) + 1
        
        if method not in coverage_by_method:
            coverage_by_method[method] = []
        coverage_by_method[method].append(coverage)
    
    print(f"Total products analyzed: {len(all_products)}")
    print("\nExtraction Methods:")
    
    for method, count in methods.items():
        percentage = count / len(all_products) * 100
        avg_coverage = sum(coverage_by_method[method]) / len(coverage_by_method[method])
        print(f"   {method}: {count} products ({percentage:.1f}%) - Avg coverage: {avg_coverage:.1%}")
    
    # LLM enhancement effectiveness
    if "hybrid" in methods:
        hybrid_coverage = coverage_by_method["hybrid"]
        pattern_coverage = coverage_by_method.get("pattern_matching", [0])
        
        print(f"\n🎯 LLM Enhancement Effectiveness:")
        print(f"   Pattern-only average: {sum(pattern_coverage)/len(pattern_coverage):.1%}")
        print(f"   Hybrid (LLM enhanced) average: {sum(hybrid_coverage)/len(hybrid_coverage):.1%}")
        
        improvement = (sum(hybrid_coverage)/len(hybrid_coverage)) - (sum(pattern_coverage)/len(pattern_coverage))
        print(f"   Improvement from LLM: {improvement:+.1%}")

analyze_llm_enhancement_usage()

## Final Summary & Next Steps

🎉 **Congratulations!** You've successfully tested the complete 4-agent pipeline:

### ✅ **What We've Tested:**
1. **QueryOrchestrator** - Intent recognition and query parsing
2. **TavilyRetriever** - Real web search and content extraction  
3. **CredibilityFilter** - Multi-factor credibility scoring and filtering
4. **SpecExtractor** - Universal product specification extraction

### 🎯 **Key SpecExtractor Findings:**
- **Universal Category Detection**: Works across electronics, kitchen, fashion, toys, automotive, books
- **Dynamic Specification Extraction**: Adapts to any product type automatically
- **Pattern Recognition + LLM Enhancement**: Comprehensive extraction approach
- **Coverage Analysis**: Measures and improves extraction quality
- **Unit Normalization**: Standardizes measurements across categories

### ➡️ **Next Agent: ResultsRankerAgent**
The pipeline is ready for the final component that will:
- Score products by relevance, price value, and quality
- Rank structured products for final presentation
- Apply user preference weighting
- Generate explanations for rankings

The SpecExtractor foundation is solid and ready for the final ranking phase! 🚀